In [1]:
import random

N = 100
random.seed(42)

OBJECTS = {n for n in range(N)}
SETS = tuple(frozenset(random.sample(range(N), k=s+1)) for s in range(N))
COSTS = tuple(10*random.random()+(s*random.random())**2 for s in range(N))

PENALTY_WEIGHT = sum(COSTS) + 1

In [2]:
def fitness(state, objects, sets, costs, penalty_weight):
    total_cost = sum(costs[idx] for idx in state)
    covered = set().union(*(sets[idx] for idx in state))

    uncovered_count = len(objects - covered)
    penalty = uncovered_count * penalty_weight

    return -(total_cost + penalty), uncovered_count == 0

In [3]:
def tweak(state, n, p_add=0.35, p_drop=0.35):
    state_set = set(state)
    available = set(range(n)) - state_set
    r = random.random()

    # 1. ADD: aggiunge un set non ancora selezionato
    if r < p_add and available:
        picked = random.choice(tuple(available))
        state_set.add(picked)

    # 2. DROP: rimuove un set attivo (solo se ne abbiamo almeno 2)
    elif r < p_add + p_drop and len(state_set) > 1:
        picked = random.choice(tuple(state_set))
        state_set.remove(picked)

    # 3. SWAP: sostituisce un set presente con uno assente
    elif state_set and available:
        drop_picked = random.choice(tuple(state_set))
        add_picked = random.choice(tuple(available))
        state_set.remove(drop_picked)
        state_set.add(add_picked)

    return frozenset(state_set)

In [4]:
def hill_climbing_sets(max_evals_without_improvement=400, initial_cardinality=5, restarts=10):
    best_overall_state = None
    best_overall_fitness = -float("inf")
    best_is_valid = False

    k = random.randint(1, initial_cardinality)
    current_state = frozenset(random.sample(range(N), k=k))
    current_fit, current_valid = fitness(current_state, OBJECTS, SETS, COSTS, PENALTY_WEIGHT)

    evals = 0
    steps = 0
    while evals < max_evals_without_improvement:
        neighbor = tweak(current_state, N)
        neighbor_fit, neighbor_valid = fitness(neighbor, OBJECTS, SETS, COSTS, PENALTY_WEIGHT)
        steps += 1

        if neighbor_fit > current_fit:
            current_state = neighbor
            current_fit = neighbor_fit
            current_valid = neighbor_valid
            evals = 0
            print(steps, best_overall_fitness, best_overall_state)
        else:
            evals += 1

        if current_fit > best_overall_fitness:
            best_overall_fitness = current_fit
            best_overall_state = current_state
            best_is_valid = current_valid

    real_cost = sum(COSTS[i] for i in best_overall_state) if best_is_valid else None
    return sorted(best_overall_state), real_cost, best_is_valid

In [5]:
best_indices, best_cost, is_valid = hill_climbing_sets()

print(f"Soluzione valida: {is_valid}")
print(f"Indici dei set scelti: {best_indices}")
print(f"Numero di set utilizzati: {len(best_indices)}")
print(f"Costo reale della soluzione: {best_cost:.4f}")

1 -inf None
2 -7596428.01306197 frozenset({18})
3 -2345875.860959141 frozenset({74})
4 -2064928.565730162 frozenset({74, 21})
9 -1596186.4252796506 frozenset({24, 74, 21})
10 -192616.32730484928 frozenset({24, 74, 80, 21})
11 -192369.69837765553 frozenset({80, 24, 74, 22})
21 -5011.42078638371 frozenset({80, 22, 54, 24, 74})
22 -4856.763972003897 frozenset({80, 24, 74, 54})
52 -4684.602804928001 frozenset({80, 74, 54})
55 -3393.5215688757908 frozenset({80, 99, 54})
60 -3186.078513112576 frozenset({80, 99})
63 -140.84699307264538 frozenset({27, 99})
Soluzione valida: True
Indici dei set scelti: [99]
Numero di set utilizzati: 1
Costo reale della soluzione: 13.2716
